In [ ]:
#!pip --version

pip 26.2.1 from D:\in0902\ex0914\.venv\Lib\site-packages\pip (python 3.12)



In [6]:
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
#!pip install langchain_classic

In [8]:
# 오류 ModuleNotFoundError
# from output_parsers.output_parsers import ResponseSchema, StructuredOutputParser
# 수정제안
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

In [9]:
# 사용자의 질문에 대한 답변
response_schemas = [
    ResponseSchema(name="answer", description="사용자의 질문에 대한 답변"),
    ResponseSchema(
        name="source",
        description="사용자의 질문에 답하기 위해 사용된 `출처`, `웹사이트주소` 이여야 합니다.",
    ),
]

In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
response = llm.invoke("LangSmith 연동 테스트입니다. 짧게 답해주세요.")
print(response.content)


네, 어떤 질문이든 말씀해 주세요!


In [11]:
# 응답 스킴마를 기반으로 한 구조화된 출력 파서 초기화
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [12]:
print(output_parser.get_format_instructions())

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"answer": string  // 사용자의 질문에 대한 답변
	"source": string  // 사용자의 질문에 답하기 위해 사용된 `출처`, `웹사이트주소` 이여야 합니다.
}
```


In [13]:
# 출력 형식 지시사항을 파싱합니다.
format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    # 사용자의 질문에 최대한 답변하도록 템플릿을 설정합니다.
    template="answer the users question as best as possible.\n{format_instructions}\n{question}",
    # 입력 변수로 'question'을 사용합니다.
    input_variables=["question"],
    # 부분 변수로 'format_instructions'을 사용합니다.
    partial_variables={"format_instructions": format_instructions},
)

In [15]:
model= ChatOpenAI(temperature=0) # ChatOpenAI 모델 초기화
chain = prompt|model|output_parser #프롬프트 , 모델 , 출력 파서를 연결